# Get to Know a Dataset: Artemis and Image Safari

This notebook introduces **Artemis and Image Safari - Crop Imagery Datasets Spanning Globally Critical Food Species**, a unified agricultural image dataset of 7,469,218 field photographs comprising two datasets: **Artemis** (1,387,663 structured phenotyping images across four crops) and **Image Safari** (6,081,555 diversity-driven images across 18 crops). This walkthrough shows how the collection is organized, how to browse it anonymously on Amazon S3, how to load an image, and how to examine the Image Safari crop distribution.

Soybean imagery is included in Artemis but has no published annotation sets in the current inventory. Finger millet and lentil are present in the Image Safari image corpus but have no published annotation sets in the current inventory. African yam (*Dioscorea* spp.) is a single documented crop; S3 prefixes `african-yam/` and `yam/` are treated as alternate paths for that crop.


## How is the dataset organized?

The collection is organized into two top-level datasets — `Artemis/` and `ImageSafari/` — each grouped by crop. Every crop directory contains an `images/` subtree, an `annotations/` subtree, and a `metadata/` subtree. Artemis annotations are nested by optional variety, task (`instance_segmentation` or `object_detection`), and named set. Image Safari annotations are under a `standard/` track by modality and named set. A subset of **44,820** images is annotated (14,250 Artemis; 30,570 Image Safari).

```text
alliance-artemis-imagesafari/
├── Artemis/
│   └── <crop>/images/...  annotations/[<variety>/]<task>/<set>/...  metadata/...
└── ImageSafari/
    └── <crop>/images/...  annotations/standard/<modality>/<set>/...  metadata/...
```

On S3, list the top-level `Artemis/` and `ImageSafari/` prefixes. The examples below use the Image Safari dataset. Object key depth can vary across contributing centres, so applications should parse keys defensively.


## Libraries and public S3 connection

This tutorial uses `boto3`, `Pillow`, and `matplotlib`. The collection is hosted at `s3://alliance-artemis-imagesafari/` in `us-west-2`.


In [ ]:
from io import BytesIO

import boto3
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config
from PIL import Image

BUCKET = "alliance-artemis-imagesafari"
REGION = "us-west-2"

s3 = boto3.client(
    "s3",
    region_name=REGION,
    config=Config(signature_version=UNSIGNED),
)


## Browse the crop prefixes

The following request lists the top-level crop prefixes without requiring AWS credentials.


In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix="ImageSafari/", Delimiter="/")
crop_prefixes = [item["Prefix"] for item in response.get("CommonPrefixes", [])]
crop_prefixes


## What formats are present?

The collection contains JPEG, PNG, and WebP still images. Pillow can decode all three formats. File extensions should be handled case-insensitively, and production pipelines should catch decode errors even though the release has undergone corruption screening.


## Download and display one image

This example finds the first supported image under the `potato/` prefix, downloads it into memory, and displays it.


In [ ]:
extensions = (".jpg", ".jpeg", ".png", ".webp")
page = s3.list_objects_v2(Bucket=BUCKET, Prefix="ImageSafari/potato/images/", MaxKeys=1000)
image_key = next(
    item["Key"]
    for item in page.get("Contents", [])
    if item["Key"].lower().endswith(extensions)
)

body = s3.get_object(Bucket=BUCKET, Key=image_key)["Body"].read()
image = Image.open(BytesIO(body)).convert("RGB")

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title(image_key)
plt.axis("off")
plt.show()


## What does the collection distribution look like?

The collection is long-tailed: potato is the largest class, while soybean and cassava are the smallest. The chart below uses the verified counts from the cleaned dataset report.


In [ ]:
crop_counts = {
    "Potato": 1689917,
    "Sorghum": 730682,
    "Pigeon pea": 698011,
    "Finger millet": 593461,
    "Cowpea": 519759,
    "Sweet potato": 462894,
    "Groundnut": 411948,
    "Banana": 334732,
    "Pearl millet": 211390,
    "African yam": 117494,
    "Lentil": 78218,
    "Chickpea": 73856,
    "Rice": 53170,
    "Wheat": 37023,
    "Maize": 23348,
    "Common bean": 22295,
    "Cassava": 12120,
    "Soybean": 11237,
}

names = list(reversed(crop_counts.keys()))
counts = [crop_counts[name] for name in names]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(names, counts, color="#2b6f4e")
ax.set_xlabel("Number of images")
ax.set_ylabel("Crop")
ax.set_title("Image Safari cleaned images by crop")
ax.spines[["top", "right"]].set_visible(False)
ax.ticklabel_format(axis="x", style="plain")
plt.tight_layout()
plt.show()

print(f"Total images: {sum(crop_counts.values()):,}")


## One answered question

**How imbalanced is the crop distribution?** Potato contributes 1,689,917 images, or approximately 27.8% of the collection. The six largest classes account for most images, while several crops contribute fewer than 100,000 images. Accuracy alone may therefore conceal poor performance on smaller classes. Macro-averaged metrics and per-crop reporting are recommended.


In [ ]:
total = sum(crop_counts.values())
largest_crop = max(crop_counts, key=crop_counts.get)
largest_share = 100 * crop_counts[largest_crop] / total

print(f"Largest crop: {largest_crop}")
print(f"Images: {crop_counts[largest_crop]:,}")
print(f"Share: {largest_share:.1f}%")


## One open research question

**How well do crop-recognition models generalize to countries, centres, seasons, and capture methods that were not represented during training?** A useful study would build group-aware splits that hold out entire centres or countries, compare pretrained visual encoders, report macro F1 and per-crop recall, and measure performance separately for each held-out domain. Near-duplicate screening should be applied before splitting to reduce leakage.


## License and citation

The dataset is released under the [Creative Commons Attribution-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-sa/4.0/). Adapted material must be distributed under the same license.

Suggested citation: Mutuvi S., Guerena D., Zych M., Henday S., Girma E., Mungubariki T., Agesa B., Zochowski M., Ciolek D., Lazowik M., Chen J., Goeke L., Omwandho J., del Palma G., Malabi J., Phomebeya S., Sanena M., Marcos J. T. C., Ghandi H., Nas M., Rathore A., Mendes T., Yadav S., Adjah K. L., Woltering L., Lekasio E., Mushi B., Abraham L., Katunzi G., Remy S. L., Siyavora T., Selvaraj M., Casas J., Boukar O., Ongom P., Nakato G. V., Mwanje G., Agre P., Laporte M.-A., Asefa T., Odama R., Mukankusi C., and Wu W. (2026). *Artemis and Image Safari - Crop Imagery Datasets Spanning Globally Critical Food Species*. Registry of Open Data on AWS.
